# **WEEK-4 Data Cleaning and Preparation**

goal :
- Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Ensure numeric fields are properly typed
- Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

### Set Up

In [1]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd
import os

DATA_DIR = Path("/Users/kmaxx/Desktop/IDX-da/idx_data")

sold_df = pd.read_csv(DATA_DIR / "sold_with_rates.csv", low_memory= False)
listing_df = pd.read_csv(DATA_DIR / "listing_with_rates.csv", low_memory= False)

### Data Convertion

In [2]:
def convert_date_fields(df):
    date_cols = [
        "CloseDate",
        "PurchaseContractDate",
        "ListingContractDate",
        "ContractStatusChangeDate"
    ]

    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

In [3]:
sold_df = convert_date_fields(sold_df)
listing_df = convert_date_fields(listing_df)

### Invalid Numerical Data Removal

In [4]:
def sold_remove_invalid_numeric_values(df):
    data = df.copy()

    numeric_cols = [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    valid_mask = (
        (data["ClosePrice"] > 0) &
        (data["LivingArea"] > 0) &
        (data["DaysOnMarket"] >= 0) &
        (data["BedroomsTotal"] >= 0) &
        (data["BathroomsTotalInteger"] >= 0)
    )

    cleaned_df = data[valid_mask].copy()

    print(f"Rows before cleaning: {len(data):,}")
    print(f"Rows after cleaning:  {len(cleaned_df):,}")
    print(f"Rows removed:         {len(data) - len(cleaned_df):,}")

    return cleaned_df

In [5]:
sold_df = sold_remove_invalid_numeric_values(sold_df)

Rows before cleaning: 448,091
Rows after cleaning:  447,559
Rows removed:         532


In [6]:
def list_remove_invalid_numeric_values(df):
    data = df.copy()

    numeric_cols = [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    valid_mask = (
        (data["ListPrice"] > 0) &
        (data["LivingArea"] > 0) &
        (data["DaysOnMarket"] >= 0) &
        (data["BedroomsTotal"] >= 0) &
        (data["BathroomsTotalInteger"] >= 0)
    )

    cleaned_df = data[valid_mask].copy()

    print(f"Rows before cleaning: {len(data):,}")
    print(f"Rows after cleaning:  {len(cleaned_df):,}")
    print(f"Rows removed:         {len(data) - len(cleaned_df):,}")

    return cleaned_df

In [7]:
listing_df = list_remove_invalid_numeric_values(listing_df)

Rows before cleaning: 615,070
Rows after cleaning:  613,936
Rows removed:         1,134


### Date Consistency Checks

In [8]:
def add_date_consistency_flags(df):
    data = df.copy()

    date_cols = [
        "ListingContractDate",
        "PurchaseContractDate",
        "CloseDate",
        "ContractStatusChangeDate"
    ]

    for col in date_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors="coerce")

    data["listing_after_close_flag"] = (
        data["ListingContractDate"] > data["CloseDate"]
    )

    data["purchase_after_close_flag"] = (
        data["PurchaseContractDate"] > data["CloseDate"]
    )

    data["negative_timeline_flag"] = (
        (data["ListingContractDate"] > data["PurchaseContractDate"]) |
        (data["PurchaseContractDate"] > data["CloseDate"]) |
        (data["ListingContractDate"] > data["CloseDate"])
    )

    return data

In [9]:
sold_df = add_date_consistency_flags(sold_df)
listing_df = add_date_consistency_flags(listing_df)

#### Date consistency summary

In [10]:
def summarize_date_consistency_flags(df):
    flag_columns = [
        "listing_after_close_flag",
        "purchase_after_close_flag",
        "negative_timeline_flag"
    ]

    summary = pd.DataFrame({
        "Flag": flag_columns,
        "Count": [df[col].sum() for col in flag_columns],
        "Percentage": [
            df[col].mean() * 100
            for col in flag_columns
        ]
    })

    invalid_mask = df[flag_columns].any(axis=1)

    total_invalid = pd.DataFrame({
        "Flag": ["any_date_issue"],
        "Count": [invalid_mask.sum()],
        "Percentage": [invalid_mask.mean() * 100]
    })

    summary = pd.concat(
        [summary, total_invalid],
        ignore_index=True
    )

    summary["Percentage"] = summary["Percentage"].round(2)

    return summary

In [11]:
summarize_date_consistency_flags(sold_df)

,Flag,Count,Percentage
0,listing_after_close_flag,70,0.02
1,purchase_after_close_flag,240,0.05
2,negative_timeline_flag,531,0.12
3,any_date_issue,531,0.12


#### Remove Invalid Dates

In [12]:
def remove_inconsistent_date_rows(df):
    flag_columns = [
        "listing_after_close_flag",
        "purchase_after_close_flag",
        "negative_timeline_flag"
    ]

    invalid_mask = df[flag_columns].any(axis=1)

    data = df.loc[~invalid_mask].copy()

    data.reset_index(drop=True, inplace=True)

    return data

In [13]:
sold_df = remove_inconsistent_date_rows(sold_df)
listing_df = remove_inconsistent_date_rows(listing_df)

### Geographic Data Checks

In [14]:
def add_geographic_flags(df):
    data = df.copy()

    data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce")
    data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce")

    # Missing coordinates
    data["missing_coordinates_flag"] = (
        data["Latitude"].isna() | data["Longitude"].isna()
    )

    # Sentinel null values
    data["zero_coordinates_flag"] = (
        (data["Latitude"] == 0) | (data["Longitude"] == 0)
    )

    # California longitudes should be negative
    data["positive_longitude_flag"] = (
        data["Longitude"] > 0
    )

    # Rough California coordinate bounds
    data["implausible_coordinates_flag"] = (
        (data["Latitude"] < 32) |
        (data["Latitude"] > 42) |
        (data["Longitude"] < -125) |
        (data["Longitude"] > -114)
    )

    return data

In [15]:
sold_df = add_geographic_flags(sold_df)
listing_df = add_geographic_flags(listing_df)

#### Invalid Geographics summary

In [16]:
def summarize_geographic_flags(df):
    flag_columns = [
        "missing_coordinates_flag",
        "zero_coordinates_flag",
        "positive_longitude_flag",
        "implausible_coordinates_flag"
    ]

    summary = pd.DataFrame({
        "Flag": flag_columns,
        "Count": [df[col].sum() for col in flag_columns],
        "Percentage": [
            df[col].mean() * 100
            for col in flag_columns
        ]
    })

    summary["Percentage"] = summary["Percentage"].round(2)

    return summary

In [17]:
summarize_geographic_flags(sold_df)

,Flag,Count,Percentage
0,missing_coordinates_flag,4353,0.97
1,zero_coordinates_flag,37,0.01
2,positive_longitude_flag,31,0.01
3,implausible_coordinates_flag,99,0.02


In [18]:
summarize_geographic_flags(listing_df)

,Flag,Count,Percentage
0,missing_coordinates_flag,80787,13.17
1,zero_coordinates_flag,74,0.01
2,positive_longitude_flag,79,0.01
3,implausible_coordinates_flag,301,0.05


#### Remove Invalid Geographics

In [19]:
def remove_invalid_geographic_rows(df):
    data = df.copy()

    invalid_mask = (
        data["missing_coordinates_flag"] |
        data["zero_coordinates_flag"] |
        data["positive_longitude_flag"] |
        data["implausible_coordinates_flag"]
    )

    data = data.loc[~invalid_mask].copy()

    data.reset_index(drop=True, inplace=True)

    return data

In [20]:
sold_df = remove_invalid_geographic_rows(sold_df)
listing_df = remove_invalid_geographic_rows(listing_df)

### Postal Code Check

In [21]:
sold_postal_city = sold_df[["PostalCode", "City"]]

In [22]:
sold_postal = pd.to_numeric(
    sold_df["PostalCode"],
    errors="coerce"
)

outside_ca = sold_df.loc[~sold_postal.between(90001, 96162)].copy()
outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
210,94568-4818,Dublin
875,94551-4971,Livermore
939,94506-123,Danville
957,94591-7809,Vallejo
961,94538-3337,Fremont


In [23]:
# Remove number behind dash
sold_df["PostalCode"] = (
    sold_df["PostalCode"]
    .astype("string")
    .str.strip()
    .str.split("-").str[0]  
    .str.zfill(5)
)

In [24]:
sold_postal = pd.to_numeric(
    sold_df["PostalCode"],
    errors="coerce"
)
outside_ca = sold_df.loc[~sold_postal.between(90001, 96162)].copy()
outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
31177,86442,Bullhead City
42196,01010,Duarte
47560,85365,Outside Area (Outside Ca)
52307,19351,Canyon Country
64207,89060,Pahrump


In [25]:
# Keep only valid California ZIP-code rows
sold_df = sold_df.loc[
    sold_postal.between(90001, 96162)
].copy()

sold_df.reset_index(drop=True, inplace=True)

In [26]:
listing_postal_city =listing_df[["PostalCode", "City"]]

In [27]:
listing_postal = pd.to_numeric(
    listing_df["PostalCode"],
    errors="coerce"
)
list_outside_ca = listing_df.loc[~listing_postal.between(90001, 96162)].copy()
list_outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
2561,65965,Oroville
3050,22566,Tijuana
3141,30054,Outside Area (Outside U.S.) Foreign Country
4287,907013,Lakewood
8849,92223-6233,Beaumont


In [28]:
# Remove number after dash
listing_df["PostalCode"] = (
    listing_df["PostalCode"]
    .astype("string")
    .str.strip()
    .str.split("-").str[0]  
    .str.zfill(5)
)

In [29]:
listing_postal = pd.to_numeric(
    listing_df["PostalCode"],
    errors="coerce"
)
listing_outside_ca = listing_df.loc[~listing_postal.between(90001, 96162)].copy()
listing_outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
2561,65965,Oroville
3050,22566,Tijuana
3141,30054,Outside Area (Outside U.S.) Foreign Country
4287,907013,Lakewood
12590,22634,Outside Area (Outside Ca)


In [30]:
# Keep only valid California ZIP-code rows
listing_df = listing_df.loc[
    listing_postal.between(90001, 96162)
].copy()

listing_df.reset_index(drop=True, inplace=True)

### Columns to Remove

#### 1. Sold

In [ ]:
sold_core_variables = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "PropertySubType",
    "City",
    "CountyOrParish",
    "PostalCode",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "ParkingTotal",
    "PoolPrivateYN",
    "rate_30yr_fixed"
]
sold_secondary_variables = [
    "PropertyType",
    "MLSAreaMajor",
    "SubdivisionName",
    "AssociationFee",
    "AssociationFeeFrequency",
    "NewConstructionYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "ViewYN",
    "MainLevelBedrooms",
    "Stories",
    "Levels",
    "Flooring",
    "LotSizeAcres",
    "HighSchoolDistrict",
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "MlsStatus",
    "StateOrProvince",
    "OriginatingSystemName",
    "OriginatingSystemSubName"
    "ListOfficeName",
    "BuyerOfficeName",
    "ListAgentFullName"
]
sold_to_remove = [
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentEmail",
    "ListAgentAOR",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentAOR",
    "BuyerOfficeAOR",
    "BuyerAgencyCompensation",
    "BuyerAgencyCompensationType"
]

In [32]:
sold_df = sold_df.drop(
    columns=sold_to_remove,
    errors="ignore"
)

#### 2. Listing

In [ ]:
list_core_variables = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "PropertySubType",
    "City",
    "CountyOrParish",
    "PostalCode",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "ParkingTotal",
    "rate_30yr_fixed"
]
list_secondary_variables = [
    "PropertyType",
    "MLSAreaMajor",
    "SubdivisionName",
    "AssociationFee",
    "AssociationFeeFrequency",
    "NewConstructionYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "MainLevelBedrooms",
    "Stories",
    "Levels",
    "LotSizeAcres",
    "LotSizeArea",
    "HighSchoolDistrict",
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "MlsStatus",
    "StateOrProvince",
    "ContractStatusChangeDate",
    "UnparsedAddress",
    "ListOfficeName",
    "BuyerOfficeName",
    "ListAgentFullName"
]
identifier_variables = [
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "StreetNumberNumeric"
]
list_to_remove = [
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentEmail",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerOfficeAOR",
    "BuyerAgencyCompensation",
    "BuyerAgencyCompensationType"
    "PropertyType.1",
    "ListAgentFirstName.1",
    "DaysOnMarket.1",
    "LivingArea.1",
    "Longitude.1",
    "Latitude.1",
    "ListPrice.1",
    "ListAgentLastName.1",
    "CloseDate.1",
    "BuyerOfficeName.1",
    "UnparsedAddress.1"
]

In [34]:
listing_df = listing_df.drop(
    columns=list_to_remove,
    errors="ignore"
)

### Review

In [35]:
sold_df.shape

(442529, 61)

In [36]:
listing_df.shape

(532178, 58)

### Export

In [37]:
sold_df.to_csv(DATA_DIR / "cleaned_sold.csv", index = False)
listing_df.to_csv(DATA_DIR / "cleaned_listing.csv", index = False)